# Module 2A — Raw Activation Extraction

Runs the circuitgpt model on all prompt variations and saves **raw activation values** (continuous, not binary) for every (AST node, builtin) pair.

## Output

`activations_{input_name}.h5` containing per pair, per layer:
- **activation_sum** — sum of |activation| across all prompts (float32, [2048])
- **firing_count** — number of prompts where |activation| > 0 (int32, [2048])
- **n_prompts** — number of prompt variations processed

No thresholding is applied here. Module 2B applies epsilon and consistency thresholds to produce universal objects.

In [ ]:
# Cell 1 – Dependency setup (circuit_sparsity injection + pip installs)
import subprocess, sys, os, types, importlib.util, glob, inspect

pkgs = ["h5py", "transformers", "torch", "numpy", "pandas", "tqdm",
        "pyarrow", "huggingface_hub"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# ── Locate real gpt.py & hook_utils.py ───────────────────────────────────
tm_base     = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/openai")
tm_gpt_hits = glob.glob(os.path.join(tm_base, "*/gpt.py"))

if tm_gpt_hits:
    gpt_path  = tm_gpt_hits[0]
    hook_path = os.path.join(os.path.dirname(gpt_path), "hook_utils.py")
    print(f"Using transformers_modules cache: {os.path.dirname(gpt_path)}")
else:
    from huggingface_hub import hf_hub_download
    gpt_path  = hf_hub_download("openai/circuit-sparsity", "gpt.py")
    hook_path = hf_hub_download("openai/circuit-sparsity", "hook_utils.py")
    print("Using hf_hub_download (first run)")

hf_dir = os.path.dirname(gpt_path)

cs_pkg = types.ModuleType("circuit_sparsity")
cs_pkg.__path__ = [hf_dir]
cs_pkg.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity"] = cs_pkg

hook_spec = importlib.util.spec_from_file_location("circuit_sparsity.hook_utils", hook_path)
hook_mod  = importlib.util.module_from_spec(hook_spec)
hook_mod.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity.hook_utils"] = hook_mod
hook_spec.loader.exec_module(hook_mod)
cs_pkg.hook_utils = hook_mod

gpt_spec = importlib.util.spec_from_file_location("circuit_sparsity.gpt", gpt_path)
gpt_mod  = importlib.util.module_from_spec(gpt_spec)
gpt_mod.__package__ = "circuit_sparsity"
sys.modules["circuit_sparsity.gpt"] = gpt_mod
gpt_spec.loader.exec_module(gpt_mod)

_RealGPTConfig = gpt_mod.GPTConfig
_valid_params  = set(inspect.signature(_RealGPTConfig.__init__).parameters) - {"self"}

class _PatchedGPTConfig(_RealGPTConfig):
    def __init__(self, **kwargs):
        super().__init__(**{k: v for k, v in kwargs.items() if k in _valid_params})

gpt_mod.GPTConfig = _PatchedGPTConfig
cs_pkg.gpt = gpt_mod

print(f"circuit_sparsity ready — GPTConfig patched")

In [ ]:
# Cell 2 – Configuration
import os

MODEL_ID       = "openai/circuit-sparsity"
PARQUET_FILE   = "prompts_115x100.parquet"

# ── Detect environment ───────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

PARQUET_PATH   = f"{DATA_DIR}/{PARQUET_FILE}"
CHECKPOINT_DIR = f"{DATA_DIR}/checkpoints"

# ── Output name derived from input ───────────────────────────────────────
INPUT_NAME     = os.path.splitext(PARQUET_FILE)[0]
OUTPUT_HDF5    = f"{DATA_DIR}/activations_{INPUT_NAME}.h5"

N_LAYERS            = 8
CHECKPOINT_EVERY    = 200
RESUME_CKPT         = None     # set to path string to resume

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Config OK")
print(f"  Data dir: {DATA_DIR}")
print(f"  Input   : {PARQUET_PATH}")
print(f"  Output  : {OUTPUT_HDF5}")

In [ ]:
# Cell 3 – Imports & Drive mount
import os, sys, logging, subprocess, shutil
import numpy as np, pandas as pd, torch, h5py
from tqdm.auto import tqdm

# Mount Drive (Colab only)
if IN_COLAB:
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)

# sys.path
LOCAL_SRC = "/Users/piotrwilam/Code/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH  = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("module2A")
print("Imports OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# Cell 4 – Load data & model
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df)} rows | "
      f"{df.groupby(['ast_node','builtin_obj']).ngroups} unique pairs")

from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, dtype=torch.float16,
).to(DEVICE).eval()

print(f"Model loaded on {DEVICE} | Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 5 – Register hooks & sanity check
import re
from module2.extraction import ActivationExtractor

print("Scanning model.named_modules() for MLP layers...")
mlp_pattern = None
seen_lids = set()

for name, _ in model.named_modules():
    m = re.match(r'^(.*?)(\d+)(\.mlp)$', name)
    if m:
        lid = int(m.group(2))
        if lid not in seen_lids:
            seen_lids.add(lid)
            print(f"  [{lid}] {name}")
        if mlp_pattern is None:
            mlp_pattern = m.group(1) + "{layer_id}" + m.group(3)

if mlp_pattern is None:
    for name, _ in model.named_modules():
        print(f"  {name}")
    raise RuntimeError("Set mlp_pattern manually and re-run.")

print(f"\nPattern: {mlp_pattern} | Layers: {sorted(seen_lids)}")

extractor = ActivationExtractor(
    model=model, tokenizer=tokenizer, device=DEVICE,
    n_layers=N_LAYERS, use_hook_recorder=False,
)
extractor.set_hook_pattern(mlp_pattern)
extractor.register_hooks()

# Sanity check
sample = df["prompt_text"].iloc[0]
acts = extractor.extract(sample)
for lid, vec in sorted(acts.items()):
    print(f"  Layer {lid}: shape={tuple(vec.shape)}, "
          f"mean |act|={vec.abs().mean().item():.6f}")

In [ ]:
# Cell 6 – Full raw activation extraction (with checkpointing)
from module2.binarization import RawActivationCollector
from module2.io_utils import save_activations_hdf5
import pickle

collector = RawActivationCollector(n_layers=N_LAYERS)
grouped = df.groupby(["ast_node", "builtin_obj"])
all_pairs = list(grouped.groups.keys())

pair_activations = {}
start_idx = 0

# Resume from checkpoint if available
if RESUME_CKPT and os.path.exists(RESUME_CKPT):
    with open(RESUME_CKPT, "rb") as f:
        ckpt = pickle.load(f)
    pair_activations = ckpt["pair_activations"]
    start_idx = ckpt["pairs_processed"]
    print(f"Resumed from checkpoint: {start_idx} pairs done")

for idx, (ast_n, blt_o) in enumerate(
    tqdm(all_pairs[start_idx:], desc="Extracting raw activations")
):
    prompts = df.loc[grouped.groups[(ast_n, blt_o)], "prompt_text"].tolist()
    raw = collector.collect(extractor, prompts)
    pair_activations[(ast_n, blt_o)] = raw

    actual_idx = idx + start_idx + 1
    if actual_idx % CHECKPOINT_EVERY == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"module2A_ckpt_{actual_idx}.pkl")
        with open(ckpt_path, "wb") as f:
            pickle.dump({"pair_activations": pair_activations,
                         "pairs_processed": actual_idx}, f)
        logger.info(f"Checkpoint: {ckpt_path}")

print(f"\nExtracted {len(pair_activations)} pairs")

In [ ]:
# Cell 7 – Save activations HDF5
metadata = {
    "model_id"    : MODEL_ID,
    "n_layers"    : N_LAYERS,
    "n_pairs"     : len(pair_activations),
    "input_file"  : os.path.basename(PARQUET_PATH),
    "ast_nodes"   : sorted(set(a for a, _ in pair_activations)),
    "builtin_objs": sorted(set(b for _, b in pair_activations)),
}

save_activations_hdf5(OUTPUT_HDF5, pair_activations, metadata)
print(f"Saved: {OUTPUT_HDF5}")

# Quick verification
import h5py
with h5py.File(OUTPUT_HDF5, "r") as f:
    def _walk(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: {obj.shape} {obj.dtype}")
    f.visititems(_walk)

In [ ]:
# Cell 8 – Cleanup
extractor.remove_hooks()
print("Hooks removed.")
print(f"\nModule 2A extraction complete.")
print(f"  Output : {OUTPUT_HDF5}")
print(f"  Pairs  : {len(pair_activations)}")
print(f"\nNext: run module2B_universals.ipynb to apply thresholds and produce universal objects.")